# SAM 3 PlantVillage Lesion Pseudo-Masks

> **Important:** PlantVillage contains laboratory imagery. SAM 3 is not disease-aware, so these are noisy pseudo-labels, not human annotations. Validate them before training and keep them separate from validation/test labels.


## 1. Configuration
All editable configuration is kept in this cell.

In [2]:
%pip install -q -U "transformers>=4.57.0" datasets accelerate safetensors opencv-python-headless \
     pandas matplotlib seaborn tqdm pyyaml huggingface_hub

In [3]:

# Install separately in Colab if necessary:
# !pip install -q -U "transformers>=4.57.0" datasets accelerate safetensors opencv-python-headless \
#     pandas matplotlib seaborn tqdm pyyaml huggingface_hub

from pathlib import Path
import os, random, time, json, shutil, warnings
import numpy as np
import pandas as pd
import torch
from PIL import Image

DATASET_ID = "geraldmc/plantvillage-full"
REPO_URL = ""  # Optional repository containing configs/classes.yaml and configs/mappings/plantvillage.yaml

SAM3_MODEL_ID = "facebook/sam3"
PILOT = True
N_PER_CLASS_PILOT = 20

LESION_PROMPTS = ["leaf spot", "diseased area", "lesion"]
LEAF_PROMPT = "leaf"
SCORE_THRESHOLD = 0.25
MIN_LESION_FRAC = 0.0005
MAX_LESION_FRAC = 0.40
MIN_COMPONENT_PIXELS = 20
SEED = 42

OUT_DIR = Path("/content/drive/MyDrive/plant-disease/pseudo_masks/plantvillage")
IMAGE_DIR = OUT_DIR / "images"
LESION_DIR = OUT_DIR / "masks_lesion"
LEAF_DIR = OUT_DIR / "masks_leaf"
OVERLAY_DIR = OUT_DIR / "overlays"
METADATA_PATH = OUT_DIR / "metadata.csv"
MANIFEST_PATH = OUT_DIR / "manifest.csv"
REVIEW_PATH = OUT_DIR / "review.csv"

PLANTSEG_DIR = ""  # Optional PlantSeg test-mask directory

for d in [OUT_DIR, IMAGE_DIR, LESION_DIR, LEAF_DIR, OVERLAY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

CLASS_MAP = {
    "Apple___Apple_scab": "apple__apple_scab",
    "Apple___Black_rot": "apple__black_rot",
    "Apple___Cedar_apple_rust": "apple__cedar_apple_rust",
    "Apple___healthy": "apple__healthy",
    "Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot": "corn__cercospora_leaf_spot_gray_leaf_spot",
    "Corn_(maize)___Common_rust_": "corn__common_rust",
    "Corn_(maize)___Northern_Leaf_Blight": "corn__northern_leaf_blight",
    "Corn_(maize)___healthy": "corn__healthy",
    "Tomato___Bacterial_spot": "tomato__bacterial_spot",
    "Tomato___Early_blight": "tomato__early_blight",
    "Tomato___Late_blight": "tomato__late_blight",
    "Tomato___Leaf_Mold": "tomato__leaf_mold",
    "Tomato___Septoria_leaf_spot": "tomato__septoria_leaf_spot",
    "Tomato___Spider_mites Two-spotted_spider_mite": "tomato__spider_mites_two_spotted_spider_mite",
    "Tomato___Target_Spot": "tomato__target_spot",
    "Tomato___Tomato_Yellow_Leaf_Curl_Virus": "tomato__tomato_yellow_leaf_curl_virus",
    "Tomato___Tomato_mosaic_virus": "tomato__tomato_mosaic_virus",
    "Tomato___healthy": "tomato__healthy",
}
HEALTHY_CLASSES = {"apple__healthy", "corn__healthy", "tomato__healthy"}

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


## 2. Mount Drive, authenticate, and load SAM 3
The Hugging Face token is read only from Colab Secrets.

In [4]:

from google.colab import drive, userdata
drive.mount("/content/drive")

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets and accept the facebook/sam3 terms on Hugging Face.")

from transformers import Sam3Model, Sam3Processor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float16 if device.type == "cuda" else torch.float32

print("Device:", device)
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID, token=HF_TOKEN)
model = Sam3Model.from_pretrained(
    SAM3_MODEL_ID,
    token=HF_TOKEN,
    torch_dtype=dtype,
).to(device).eval()

print("Loaded:", SAM3_MODEL_ID)


MessageError: [dfs_ephemeral] Credentials propagation unsuccessful: the server still reports the credentials as unauthorized

## 3. Dataset loading and class selection
The loader tolerates missing optional columns and keeps the original row index for stable image IDs.

In [ ]:

from datasets import load_dataset, DatasetDict
from tqdm.auto import tqdm

def load_plantvillage():
    try:
        ds = load_dataset(DATASET_ID)
    except Exception as exc:
        raise RuntimeError(
            "Dataset loading failed. Try DATASET_ID='mohanty/PlantVillage' or inspect its available configs."
        ) from exc
    if isinstance(ds, DatasetDict):
        split = "train" if "train" in ds else list(ds.keys())[0]
        return ds[split]
    return ds

dataset = load_plantvillage()
print(dataset)
print("Columns:", dataset.column_names)

label_column = "class_label" if "class_label" in dataset.column_names else None
if label_column is None:
    raise KeyError("Expected class_label column was not found.")

selected_indices = [
    i for i, row in enumerate(dataset)
    if row.get(label_column) in CLASS_MAP
]

if PILOT:
    by_class = {}
    for i in selected_indices:
        key = CLASS_MAP[dataset[i][label_column]]
        by_class.setdefault(key, []).append(i)
    selected_indices = [
        i for key in sorted(by_class)
        for i in random.sample(by_class[key], min(N_PER_CLASS_PILOT, len(by_class[key])))
    ]

print("Selected images:", len(selected_indices))


## 4. SAM 3 inference helpers
SAM 3 is called through the official Transformers `Sam3Model` and `Sam3Processor` APIs. Each prompt is evaluated independently.

In [ ]:

def image_from_row(row):
    value = row.get("image")
    if isinstance(value, Image.Image):
        return value.convert("RGB")
    if isinstance(value, dict) and "bytes" in value:
        from io import BytesIO
        return Image.open(BytesIO(value["bytes"])).convert("RGB")
    if isinstance(value, str):
        return Image.open(value).convert("RGB")
    raise TypeError("Unsupported image column value.")

def sam_prompt(image, prompt):
    inputs = processor(
        images=image,
        text=prompt,
        return_tensors="pt",
    )
    inputs = {k: v.to(device) if hasattr(v, "to") else v for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)

    processed = processor.post_process_instance_segmentation(
        outputs,
        threshold=SCORE_THRESHOLD,
        mask_threshold=0.5,
        target_sizes=[image.size[::-1]],
    )[0]

    masks = processed.get("masks", torch.empty((0, image.height, image.width)))
    scores = processed.get("scores", torch.empty((0,)))
    return masks.detach().cpu().numpy().astype(bool), scores.detach().cpu().numpy()

def infer_masks(image, lesion_prompts=LESION_PROMPTS):
    leaf_masks, leaf_scores = sam_prompt(image, LEAF_PROMPT)
    if len(leaf_masks):
        leaf = leaf_masks[np.argmax(leaf_scores)]
    else:
        leaf = np.zeros((image.height, image.width), dtype=bool)

    lesion = np.zeros_like(leaf, dtype=bool)
    all_scores = []
    for prompt in lesion_prompts:
        masks, scores = sam_prompt(image, prompt)
        all_scores.extend(scores.tolist())
        for mask, score in zip(masks, scores):
            if score >= SCORE_THRESHOLD:
                lesion |= mask

    lesion &= leaf
    return leaf, lesion, all_scores

def remove_small_components(mask, minimum_pixels):
    import cv2
    n, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask.astype(np.uint8), connectivity=8
    )
    result = np.zeros_like(mask, dtype=bool)
    for component in range(1, n):
        if stats[component, cv2.CC_STAT_AREA] >= minimum_pixels:
            result[labels == component] = True
    return result

def quality_metrics(leaf, lesion, scores, class_name):
    leaf_pixels = int(leaf.sum())
    lesion = remove_small_components(lesion, MIN_COMPONENT_PIXELS)
    lesion_pixels = int(lesion.sum())
    frac = lesion_pixels / leaf_pixels if leaf_pixels else 0.0
    leaf_frac = leaf_pixels / leaf.size
    n_components = 0

    import cv2
    if lesion_pixels:
        n_components = cv2.connectedComponents(lesion.astype(np.uint8))[0] - 1

    reasons = []
    if leaf_pixels == 0:
        reasons.append("no_leaf")
    if lesion_pixels == 0:
        reasons.append("empty_mask")
    if frac < MIN_LESION_FRAC and class_name not in HEALTHY_CLASSES:
        reasons.append("too_small")
    if frac > MAX_LESION_FRAC:
        reasons.append("too_large")
    if class_name in HEALTHY_CLASSES and frac > MIN_LESION_FRAC:
        reasons.append("healthy_with_lesion")

    return lesion, {
        "lesion_area_frac": frac,
        "leaf_area_frac": leaf_frac,
        "n_components": n_components,
        "max_sam_score": max(scores) if scores else 0.0,
        "qc_pass": not reasons,
        "qc_reason": "|".join(reasons) if reasons else "ok",
    }


## 5. SAM 3 smoke test
This verifies output keys, tensor shapes, scores, and the post-processing function before batch generation.

In [ ]:

test_image = image_from_row(dataset[selected_indices[0]])
test_masks, test_scores = sam_prompt(test_image, LESION_PROMPTS[0])
print("Processor:", type(processor).__name__)
print("Model:", type(model).__name__)
print("Post-processing function:", processor.post_process_instance_segmentation)
print("Mask shape:", test_masks.shape)
print("Scores shape:", test_scores.shape)
print("Scores:", test_scores)


## 6. Prompt experiment
Eight images from different classes are displayed for visual prompt selection.

In [ ]:

import matplotlib.pyplot as plt

experiment_indices = []
seen = set()
for i in selected_indices:
    cls = CLASS_MAP[dataset[i][label_column]]
    if cls not in seen:
        experiment_indices.append(i)
        seen.add(cls)
    if len(experiment_indices) == 8:
        break

fig, axes = plt.subplots(len(experiment_indices), 2, figsize=(10, 4 * len(experiment_indices)))
if len(experiment_indices) == 1:
    axes = np.array([axes])

for row_num, index in enumerate(experiment_indices):
    image = image_from_row(dataset[index])
    leaf, lesion, _ = infer_masks(image)
    axes[row_num, 0].imshow(image)
    axes[row_num, 0].set_title(CLASS_MAP[dataset[index][label_column]])
    axes[row_num, 1].imshow(image)
    axes[row_num, 1].imshow(lesion, alpha=0.55, cmap="Reds")
    axes[row_num, 1].imshow(~leaf, alpha=0.20, cmap="Blues")
    axes[row_num, 1].set_title("Lesion overlay / leaf boundary")
    for ax in axes[row_num]:
        ax.axis("off")
plt.tight_layout()


## 7. Resumable pseudo-mask generation
Each image is saved immediately. Metadata is checkpointed every 200 images and failures are logged.

In [ ]:

from PIL import ImageDraw
import csv

existing = set()
if METADATA_PATH.exists():
    existing = set(pd.read_csv(METADATA_PATH, usecols=["image_id"])["image_id"])

records = []
failures = []
started = time.time()

for position, index in enumerate(tqdm(selected_indices, desc="Generating pseudo-masks")):
    row = dataset[index]
    image_id = f"plantvillage_{index:06d}"
    class_name = CLASS_MAP[row[label_column]]

    if image_id in existing:
        continue

    try:
        image = image_from_row(row).convert("RGB")
        leaf, lesion, scores = infer_masks(image)
        lesion, qc = quality_metrics(leaf, lesion, scores, class_name)

        image_path = IMAGE_DIR / f"{image_id}.jpg"
        lesion_path = LESION_DIR / f"{image_id}.png"
        leaf_path = LEAF_DIR / f"{image_id}.png"
        overlay_path = OVERLAY_DIR / f"{image_id}.jpg"

        image.save(image_path, quality=95)
        Image.fromarray(lesion.astype(np.uint8)).save(lesion_path)
        Image.fromarray(leaf.astype(np.uint8)).save(leaf_path)

        overlay = np.asarray(image).copy()
        overlay[lesion] = (0.55 * overlay[lesion] + 0.45 * np.array([255, 0, 0])).astype(np.uint8)
        Image.fromarray(overlay).save(overlay_path, quality=92)

        record = {
            "image_id": image_id,
            "image_path": str(image_path),
            "lesion_mask_path": str(lesion_path),
            "leaf_mask_path": str(leaf_path),
            "overlay_path": str(overlay_path),
            "class_name": class_name,
            "crop": class_name.split("__")[0],
            "original_split": row.get("split", ""),
            "leaf_id": row.get("leaf_id", ""),
            "source": "pseudo_sam3",
            "sam_model_id": SAM3_MODEL_ID,
            "prompts_used": json.dumps(LESION_PROMPTS),
            **qc,
        }
        records.append(record)

        if len(records) >= 200:
            old = pd.read_csv(METADATA_PATH) if METADATA_PATH.exists() else pd.DataFrame()
            pd.concat([old, pd.DataFrame(records)], ignore_index=True).to_csv(METADATA_PATH, index=False)
            records.clear()

        if device.type == "cuda" and position % 50 == 0:
            torch.cuda.empty_cache()

    except Exception as exc:
        failures.append({"image_id": image_id, "error": repr(exc)})
        print(f"FAILED {image_id}: {exc}")

if records:
    old = pd.read_csv(METADATA_PATH) if METADATA_PATH.exists() else pd.DataFrame()
    pd.concat([old, pd.DataFrame(records)], ignore_index=True).to_csv(METADATA_PATH, index=False)

if failures:
    pd.DataFrame(failures).to_csv(OUT_DIR / "failures.csv", index=False)

print("Runtime minutes:", round((time.time() - started) / 60, 2))


## 8. Healthy-class sanity check and diseased-class empty-mask rates

In [ ]:

metadata = pd.read_csv(METADATA_PATH)

def class_summary(group):
    healthy = group.name in HEALTHY_CLASSES
    if healthy:
        rate = (group["lesion_area_frac"] > MIN_LESION_FRAC).mean()
        return pd.Series({"healthy_false_positive_rate": rate})
    return pd.Series({"diseased_empty_mask_rate": (group["qc_reason"].str.contains("empty_mask")).mean()})

print(metadata.groupby("class_name").apply(class_summary))


## 9. Review contact sheets and human review file

In [ ]:

def contact_sheet(frame, title, output_name, n=24):
    frame = frame.sample(min(n, len(frame)), random_state=SEED) if len(frame) else frame
    if frame.empty:
        print("No samples:", title)
        return
    fig, axes = plt.subplots((len(frame) + 3) // 4, 4, figsize=(16, 4 * ((len(frame) + 3) // 4)))
    axes = np.atleast_1d(axes).ravel()
    for ax, (_, item) in zip(axes, frame.iterrows()):
        image = np.asarray(Image.open(item["image_path"]).convert("RGB"))
        mask = np.asarray(Image.open(item["lesion_mask_path"])) > 0
        ax.imshow(image)
        ax.imshow(mask, cmap="Reds", alpha=0.5)
        ax.set_title(f'{item["class_name"]}\n{item["qc_reason"]}')
        ax.axis("off")
    for ax in axes[len(frame):]:
        ax.axis("off")
    fig.suptitle(title)
    fig.tight_layout()
    fig.savefig(OUT_DIR / output_name, dpi=120)
    plt.show()

for cls, group in metadata.groupby("class_name"):
    contact_sheet(group[group.qc_pass == True], f"{cls} — passed", f"review_passed_{cls}.jpg")
    contact_sheet(group[group.qc_pass == False], f"{cls} — rejected", f"review_rejected_{cls}.jpg")

if REVIEW_PATH.exists():
    review = pd.read_csv(REVIEW_PATH)
else:
    review = metadata[["image_id"]].copy()
    review["human_ok"] = ""
    review["notes"] = ""
    review.to_csv(REVIEW_PATH, index=False)
print("Human review file:", REVIEW_PATH)


## 10. Optional PlantSeg validation — high priority
This section is skipped unless `PLANTSEG_DIR` contains paired images and masks.

In [ ]:

if not PLANTSEG_DIR:
    print("PlantSeg validation skipped: set PLANTSEG_DIR to enable it.")
else:
    print(
        "PlantSeg validation requires a crop-matched image/mask layout. "
        "Adapt the glob patterns below to the local dataset."
    )
    # Expected example:
    # PLANTSEG_DIR/images/<crop>/*.jpg
    # PLANTSEG_DIR/masks/<crop>/<same-stem>.png
    #
    # Run infer_masks(image), binarize the PlantSeg mask with mask > 0,
    # then compute:
    #
    # intersection = (prediction & truth).sum()
    # union = (prediction | truth).sum()
    # iou = intersection / union if union else 1.0
    # dice = 2 * intersection / (prediction.sum() + truth.sum())


## 11. Export manifests
The training manifest contains only passed pseudo-labels. The full manifest retains every generated item.

In [ ]:

metadata = pd.read_csv(METADATA_PATH)
export_columns = [
    "image_id", "image_path", "lesion_mask_path", "leaf_mask_path",
    "class_name", "crop", "original_split", "leaf_id", "source",
    "sam_model_id", "prompts_used", "qc_pass"
]
metadata[export_columns].to_csv(MANIFEST_PATH, index=False)
metadata[metadata["qc_pass"] == True][export_columns].to_csv(
    OUT_DIR / "manifest_training.csv", index=False
)
print("Full manifest:", MANIFEST_PATH)
print("Training manifest:", OUT_DIR / "manifest_training.csv")


## 12. Final summary

In [ ]:

metadata = pd.read_csv(METADATA_PATH)
print("Generated:", len(metadata))
print("Passed:", int(metadata["qc_pass"].sum()))
print("Rejected:", int((~metadata["qc_pass"]).sum()))

summary = metadata.groupby("class_name").agg(
    generated=("image_id", "count"),
    passed=("qc_pass", "sum"),
    mean_lesion_area_frac=("lesion_area_frac", "mean"),
)
summary["rejected"] = summary["generated"] - summary["passed"]
print(summary)

print("\nRejection reasons:")
print(metadata.loc[~metadata.qc_pass, "qc_reason"].value_counts())
